# Telescope Resource Catalog construction decisions (B)

This notebook records only the decisions required to construct the Telescope Resource Catalog. It asks what telescope/instrument resources exist in ICARE and what static or semi-static characteristics are known about them. Its inputs are the ICARE-derived telescope and instrument tables plus `data/raw/reference/telescope_external_capabilities.csv`; it writes no data product.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 100)

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/interim/telescopes").is_dir()
)
capture_dirs = sorted((ROOT / "data/interim/telescopes").glob("capture_*"))
if not capture_dirs:
    raise RuntimeError("No ICARE telescope interim capture is available")
CAPTURE_DIR = capture_dirs[-1]
EXTERNAL_PATH = ROOT / "data/raw/reference/telescope_external_capabilities.csv"

INPUTS_READ = {
    "telescopes": CAPTURE_DIR / "telescopes.parquet",
    "instruments": CAPTURE_DIR / "instruments.parquet",
    "external_capabilities": EXTERNAL_PATH,
}

telescopes = pd.read_parquet(INPUTS_READ["telescopes"])
instruments = pd.read_parquet(INPUTS_READ["instruments"])
external = pd.read_csv(EXTERNAL_PATH, dtype=str, keep_default_na=False)

print(f"ICARE capture: {CAPTURE_DIR.name}")
print(f"telescopes={len(telescopes)}, instruments={len(instruments)}")
print(f"curated external rows={len(external)}")
print("inputs read:")
for name, path in INPUTS_READ.items():
    print(f"  {name}: {path.relative_to(ROOT)}")

In [ ]:
def normalize_label(value: str) -> str:
    return str(value).strip().casefold()

icare_pairs = instruments[["telescope.name", "name", "telescope_id", "id"]].copy()
icare_pairs.columns = ["icare_telescope", "icare_instrument", "telescope_id", "instrument_id"]

external_keys = list(zip(external["icare_telescope"], external["icare_instrument"]))
icare_keys = set(zip(icare_pairs["icare_telescope"], icare_pairs["icare_instrument"]))
exact_match_count = sum(key in icare_keys for key in external_keys)

external = external.assign(
    normalized_telescope=external["icare_telescope"].map(normalize_label),
    normalized_instrument=external["icare_instrument"].map(normalize_label),
)
icare_pairs = icare_pairs.assign(
    normalized_telescope=icare_pairs["icare_telescope"].map(normalize_label),
    normalized_instrument=icare_pairs["icare_instrument"].map(normalize_label),
)
normalized_columns = ["normalized_telescope", "normalized_instrument"]
normalized_icare_keys = set(map(tuple, icare_pairs[normalized_columns].to_numpy()))
normalized_external_keys = list(map(tuple, external[normalized_columns].to_numpy()))
normalized_match_count = sum(key in normalized_icare_keys for key in normalized_external_keys)
unique_normalized_key_count = external[normalized_columns].drop_duplicates().shape[0]
unmatched_external = external[
    ~external[normalized_columns].apply(tuple, axis=1).isin(normalized_icare_keys)
]
external_key_set = set(normalized_external_keys)
icare_without_external = icare_pairs[
    ~icare_pairs[normalized_columns].apply(tuple, axis=1).isin(external_key_set)
]["telescope_id instrument_id icare_telescope icare_instrument".split()]

known_mlim = int(external["mlim_mag"].str.strip().ne("").sum())
unknown_mlim = len(external) - known_mlim
eligibility = external["followup_eligible"].str.strip().str.casefold()
followup_eligible = int(eligibility.eq("true").sum())
followup_ineligible = int(eligibility.eq("false").sum())

summary = pd.Series({
    "CSV rows": len(external),
    "unique normalized keys": unique_normalized_key_count,
    "exact ICARE matches": exact_match_count,
    "normalized ICARE matches": normalized_match_count,
    "unmatched CSV rows": len(unmatched_external),
    "ICARE instruments without CSV rows": len(icare_without_external),
    "known Mlim": known_mlim,
    "unknown Mlim": unknown_mlim,
    "follow-up eligible": followup_eligible,
    "follow-up ineligible": followup_ineligible,
})
summary.to_frame("count")

## B1 — ICARE is the canonical resource universe

**Observed.** ICARE provides stable telescope and instrument records with native numeric IDs.

**Why it matters.** External capability metadata needs an authoritative resource universe and must not invent facilities absent from ICARE.

**Decided.** ICARE telescope and instrument identities and IDs define the catalog rows. External rows may enrich matched ICARE resources; they do not define additional resources.

**Scope.** All telescope and instrument records in the selected ICARE interim capture.

## B2 — Telescope and instrument are distinct

**Observed.** ICARE exposes telescopes and instruments as separate entities linked by `instrument.telescope_id`. One telescope can host more than one instrument.

**Why it matters.** Filters, sensitivity, and observing modes can be instrument-specific and must not be assigned indiscriminately at telescope level.

**Decided.** One catalog row represents one ICARE instrument associated with one ICARE telescope. Keep both telescope and instrument identities, using their ICARE IDs after external matching.

**Scope.** The complete ICARE telescope–instrument relation.

## B3 — External metadata source

**Observed.** The curated CSV contains the static fields needed beyond ICARE, including Mlim context, eligibility, restrictions, and provenance notes.

**Why it matters.** Catalog construction needs one explicit external input contract.

**Decided.** The sole external static capability source consumed by the catalog pipeline is `data/raw/reference/telescope_external_capabilities.csv`.

**Scope.** Every curated external value included in the Telescope Resource Catalog.

## B4 — Catalog scope

**Observed.** The selected metadata primarily describes resources relevant to optical photometry and imaging work, while ICARE also contains other resource types.

**Why it matters.** A focused catalog must not silently erase ICARE resources merely because their use falls outside the selected descriptive scope.

**Decided.** Describe the available photometry/imaging characteristics where supported. Retain out-of-scope ICARE resources with appropriate static eligibility or restriction metadata rather than silently dropping them.

**Scope.** All ICARE telescope–instrument rows represented in the catalog.

## B5 — Native ICARE fields

**Observed.** ICARE supplies native IDs, telescope and instrument names, telescope coordinates/elevation/diameter, instrument type/band and instrument filters. Diameter is populated for all 89 telescopes and band for all 95 instruments in the frozen capture.

**Why it matters.** External metadata must enrich the canonical resource records without replacing their identity or native characteristics.

**Decided.** ICARE remains authoritative for native identity and resource fields. Retain native telescope diameter as `telescope_diameter` and native instrument band as `instrument_band`, without scientific normalization or casing changes. External values must not overwrite ICARE IDs, names, type, coordinates, diameter, band or filters.

**Scope.** Every native ICARE field selected for the Telescope Resource Catalog.

## B6 — Minimal external matching

**Observed.** The curated CSV names ICARE telescopes and instruments, and its normalized name pairs are unique.

**Why it matters.** Matching should be deterministic and should not introduce ambiguous guessed associations.

**Decided.** Join external rows using stripped telescope name, stripped instrument name and case-insensitive comparison only. Remove leading/trailing whitespace from canonical names in the catalog representation; ICARE IDs remain the authoritative identity. Apply no other semantic name normalization and no fuzzy matching.

**Scope.** Every curated external row matched to the ICARE telescope–instrument universe.

## B7 — Limiting magnitude

**Observed.** The curated CSV provides Mlim for some resources and preserves filter, exposure, and source/provenance context where available. Frozen ICARE sensitivity metadata supplies an explicit native tuple for TAROT/TRE, while some external values are accepted representative network/group defaults and some reviewed numbers are spectroscopic rather than photometric.

**Why it matters.** Limiting magnitude is contextual capability metadata rather than an unconditional performance value.

**Decided.** Store only photometric/imaging Mlim as descriptive telescope/instrument capability metadata and preserve its available filter, exposure and source/provenance context. When explicit native ICARE sensitivity metadata exists, it takes precedence over a conflicting external tuple for that resource. Do not store a spectroscopic threshold as photometric Mlim. Accepted representative group/network values may remain only when provenance explicitly says they are representative and not instrument-specific measured limits. Do not apply an event-specific sensitivity decision.

**Scope.** Curated Mlim values and their accompanying static context fields.

In [ ]:
print(f"Mlim known:   {known_mlim}")
print(f"Mlim unknown: {unknown_mlim}")
print("context columns:", [
    column for column in ["mlim_filter", "mlim_exposure", "mlim_source", "provenance_note"]
    if column in external.columns
])

## B8 — Missing limiting magnitude

**Observed.** Some valid curated rows have no numeric Mlim. The absence is explicit and expected.

**Why it matters.** Absence of metadata does not establish poor performance or ineligibility.

**Decided.** Preserve missing Mlim as null with `mlim_status = UNKNOWN`. Do not infer or fabricate a value, and do not derive eligibility from its absence.

**Scope.** Every catalog row without a curated numeric Mlim value.

## B9 — Filters

**Observed.** ICARE supplies instrument-level filter lists, while a curated Mlim may be tied to a particular filter or have no filter context.

**Why it matters.** The instrument's available filters and the context of one external Mlim value describe different facts.

**Decided.** Retain ICARE instrument filters as native instrument metadata. When present, `mlim_filter` records only the band associated with the external Mlim value; do not conflate it with the instrument filter list.

**Scope.** The catalog fields `filters` and `mlim_filter`.

## B10 — Historical observations

**Observed.** ICARE observations may contain `observation.limmag`, tied to a particular instrument, filter, exposure, time, and observing conditions.

**Why it matters.** A historical achieved depth does not establish a stable nominal capability for every future observation.

**Decided.** Keep historical observations separate. Do not promote `observation.limmag` to nominal/static Mlim and do not join historical observations into the final catalog.

**Scope.** Historical ICARE observations retained in raw/interim data for possible future use.

In [ ]:
CATALOG_NATIVE_COLUMNS = {
    "telescopes": ["id", "name", "lat", "lon", "elevation", "diameter"],
    "instruments": ["id", "telescope_id", "name", "type", "band", "filters"],
}
for table_name, columns in CATALOG_NATIVE_COLUMNS.items():
    available = telescopes.columns if table_name == "telescopes" else instruments.columns
    print(f"{table_name} native catalog fields: {[column for column in columns if column in available]}")
print("excluded construction inputs: allocations, observations")

## B11 — Allocations

**Observed.** ICARE allocations connect groups or proposals to instruments and allocated observing time.

**Why it matters.** Access and authorization relationships are not static characteristics of the telescope/instrument resource itself.

**Decided.** Exclude allocations from the final static resource catalog. Retain them in raw/interim ICARE data for future work.

**Scope.** All ICARE allocation records.

## B12 — Static follow-up eligibility

**Observed.** The curated CSV provides `followup_eligible` and an optional restriction note for matched resources.

**Why it matters.** Static scientific suitability is useful catalog metadata, but it is distinct from current operational availability and scheduling.

**Decided.** Define `followup_eligible` narrowly as static suitability for targeted optical photometric/imaging follow-up. Stable false reasons include spectroscopy-only or high-energy-only scope, a generic/non-physical aggregate, or a non-repointable survey. Assess mixed missions at instrument level. The field does not represent current operational status, weather, queue state, observability, detectability or scheduling.

**Scope.** Curated eligibility and restriction fields for matched external rows.

## B13 — Dynamic information excluded

**Observed.** Observability, event-specific sensitivity, weather, queue and availability state depend on an event and/or observation time.

**Why it matters.** Persisting dynamic or event-specific results in a static catalog would make its meaning stale or ambiguous.

**Decided.** Exclude current observability, event-specific sensitivity, event magnitude, event airmass, current weather, queue state, current availability, ranking and generated application output from the Telescope Resource Catalog. Historical availability statements may remain explicitly dated in notes, but they must not determine static `followup_eligible`.

**Scope.** All event-dependent, time-dependent and generated application outputs.

In [ ]:
print(f"follow-up eligible:   {followup_eligible}")
print(f"follow-up ineligible: {followup_ineligible}")
external.loc[eligibility.eq("false"), [
    "icare_telescope", "icare_instrument", "restriction_note"
]].reset_index(drop=True)

## B14 — FoV

**Observed.** FoV is not part of the curated capability contract selected for the current catalog deliverable.

**Why it matters.** Requiring or deriving it now would expand the selected catalog scope.

**Decided.** Do not include FoV in the current normalized resource catalog. This scope choice does not imply that FoV is scientifically irrelevant.

**Scope.** The current Telescope Resource Catalog schema.

In [ ]:
print(f"exact matches:      {exact_match_count} of {len(external)}")
print(f"normalized matches: {normalized_match_count} of {len(external)}")
print(f"unique normalized keys: {unique_normalized_key_count} of {len(external)}")
print(f"unmatched external rows: {len(unmatched_external)}")
print(f"ICARE instruments without an external row: {len(icare_without_external)}")
icare_without_external.reset_index(drop=True)

## B15 — Provenance

**Observed.** Catalog rows combine native ICARE fields with selected values from the curated external CSV. Upstream expert communications are not runtime inputs, so the CSV carries their accepted concise provenance.

**Why it matters.** Users must be able to distinguish authoritative ICARE information from curated external capability metadata.

**Decided.** Preserve concise source/provenance context for curated external metadata while keeping native ICARE fields identifiable. Mark representative network/group Mlim explicitly as representative and not instrument-specific. Expert-derived values may remain accepted curated inputs even when the original communication artifact is not a runtime or repository input. Do not build a more complex provenance framework for this deliverable.

**Scope.** Every final catalog field derived from `telescope_external_capabilities.csv`.

In [ ]:
controls = [
    ("ICARE telescopes loaded", len(telescopes) > 0, str(len(telescopes))),
    ("ICARE instruments loaded", len(instruments) > 0, str(len(instruments))),
    ("curated CSV loaded", len(external) > 0, str(len(external))),
    ("only catalog construction sources were read",
     set(INPUTS_READ) == {"telescopes", "instruments", "external_capabilities"},
     ", ".join(INPUTS_READ)),
    ("required native telescope fields are present",
     set(CATALOG_NATIVE_COLUMNS["telescopes"]) <= set(telescopes.columns),
     str(CATALOG_NATIVE_COLUMNS["telescopes"])),
    ("required native instrument fields are present",
     set(CATALOG_NATIVE_COLUMNS["instruments"]) <= set(instruments.columns),
     str(CATALOG_NATIVE_COLUMNS["instruments"])),
    ("normalized external keys are unique", unique_normalized_key_count == len(external),
     f"{unique_normalized_key_count}/{len(external)}"),
    ("every external row matches ICARE after minimal normalization",
     normalized_match_count == len(external), f"{normalized_match_count}/{len(external)}"),
    ("Mlim states cover every external row", known_mlim + unknown_mlim == len(external),
     f"{known_mlim}+{unknown_mlim}={len(external)}"),
    ("eligibility states cover every external row",
     followup_eligible + followup_ineligible == len(external),
     f"{followup_eligible}+{followup_ineligible}={len(external)}"),
]
evidence_checks = pd.DataFrame([
    {"check": name, "status": "PASS" if condition else "FAIL", "detail": detail}
    for name, condition, detail in controls
])
print(evidence_checks.to_string(index=False))
failed = evidence_checks[evidence_checks["status"] == "FAIL"]
if not failed.empty:
    raise RuntimeError("Decision-notebook evidence checks failed:\n" + failed.to_string(index=False))

## Future uses explicitly out of scope

Future systems may combine the Telescope Resource Catalog with event data and observation time for observability, sensitivity filtering, detectability, current availability, scheduling or ranking. ML and LLM applications may also consume the catalog. None of those algorithms or outputs is defined or implemented here.